<a href="https://colab.research.google.com/github/vikrant48/AI-ML-python-code/blob/main/LangGraph%26tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install langchain langchain-community langchain-huggingface langchain-text-splitters langchain-core langchain-groq langgraph
!pip install chromadb sentence-transformers pypdf rank-bm25 ragas datasets

In [ ]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY").strip()
os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"]= userdata.get("LANGCHAIN_TRACING_V2")
os.environ["ANONYMIZED_TELEMETRY"] = "False"

In [ ]:
# create chunks

from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=100
)

In [ ]:
# for embedding
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en"   # better for retrieval than MiniLM
)

In [ ]:
#for base LLM
from langchain_groq import ChatGroq

base_llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.2
)

In [ ]:
import re

def extract_name(text):
    lines = text.split("\n")

    for line in lines[:5]:  # check first 5 lines
        line = line.strip()

        # simple rule: name = line with only alphabets and spaces
        if re.match(r"^[A-Za-z ]{3,}$", line):
            return line

    return "Unknown"

In [ ]:
# here for multiple resume (load pdf and chunking )
from langchain_community.document_loaders import PyPDFLoader

resume_files = [
    "resume1.pdf",
    "resume2.pdf",
    "resume3.pdf"
]
# for store chunking
chunks = []

for file in resume_files:
    loader = PyPDFLoader(file)
    docs = loader.load()

    if not docs:
        print(f"Skipping empty file: {file}")
        continue

    first_page_text = docs[0].page_content
    name = extract_name(first_page_text)
    print("Extracted Name:", name)

    for doc in docs:
        doc.metadata["name"] = name
        doc.metadata["source"] = file   # useful later

    file_chunk = splitter.split_documents(docs)

    chunks.extend(file_chunk)


In [ ]:
#create and store
from langchain_community.vectorstores import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

print("Vector DB created")

In [ ]:
#Create BM25
from rank_bm25 import BM25Okapi

corpus_texts = [chunk.page_content for chunk in chunks]
corpus_tokens = [text.lower().split() for text in corpus_texts]

bm25 = BM25Okapi(corpus_tokens)

print("BM25 ready")

In [ ]:
# hybrid search
import numpy as np
def get_dense_rankings(query, k=10):

    dense_docs = vectorstore.similarity_search(query, k=k)

    ranked_indices = []

    for doc in dense_docs:

        for i, chunk in enumerate(chunks):

            if chunk.page_content == doc.page_content:
                ranked_indices.append(i)
                break

    return ranked_indices


def get_bm25_rankings(query, k=10):

    scores = bm25.get_scores(query.lower().split())

    ranked_indices = np.argsort(scores)[::-1][:k]

    return ranked_indices.tolist()


def reciprocal_rank_fusion(rank_lists, k=60):

    scores = {}

    for rank_list in rank_lists:

        for rank, idx in enumerate(rank_list):

            scores[idx] = scores.get(idx, 0) + 1 / (k + rank + 1)

    return sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

In [ ]:
def hybrid_search(query, k=10):

    dense_rank = get_dense_rankings(query, k=10)

    bm25_rank = get_bm25_rankings(query, k=10)

    fused = reciprocal_rank_fusion([
        dense_rank,
        bm25_rank
    ])

    results = []
    seen = set()

    for idx, score in fused:

        if idx not in seen:

            results.append((chunks[idx], score))

            seen.add(idx)

        if len(results) >= k:
            break

    return results

In [ ]:
#TOOLS
from langchain_core.tools import tool

@tool
def search_candidates(query: str) -> str:
    """
    Search candidate resumes matching a job query.
    """

    results = hybrid_search(query, k=5)

    context_parts = []

    for i, (doc, score) in enumerate(results):

        meta = doc.metadata

        context_parts.append(f"""
Candidate {i+1}

Source: {meta.get('source')}
Experience: {meta.get('years')} years

Match Score: {score:.4f}

Resume:
{doc.page_content[:500]}
""")

    return "\n\n----------------\n\n".join(context_parts)

In [ ]:
#call LLM and bind tools

llm = base_llm.bind_tools([
    search_candidates
])

In [ ]:
#LANGGRAPH STATE
from typing import TypedDict, List, Annotated
from langgraph.graph.message import add_messages

class ResumeState(TypedDict):

    messages: Annotated[list, add_messages]

In [ ]:
#agent Node
def agent_node(state: ResumeState):

    response = llm.invoke(state["messages"])

    return {
        "messages": [response]
    }

In [ ]:
#Tool Node
from langgraph.prebuilt import ToolNode

tool_node = ToolNode([search_candidates])

In [ ]:
def should_continue(state: ResumeState):

    last_message = state["messages"][-1]

    if last_message.tool_calls:
        return "tools"

    return END

In [ ]:
#BUILD LANGGRAPH
from langgraph.graph import StateGraph, END
graph = StateGraph(ResumeState)

graph.add_node("agent", agent_node)

graph.add_node("tools", tool_node)

graph.set_entry_point("agent")

graph.add_conditional_edges(
    "agent",
    should_continue,
    {
        "tools": "tools",
        END: END
    }
)

graph.add_edge("tools", "agent")

app = graph.compile()

print("LangGraph app ready")

In [ ]:
#RUN QUERY
from langchain_core.messages import HumanMessage
query = """
Looking for a Java backend engineer with:
- Spring Boot
- AWS
- REST APIs
- Microservices
- SQL knowledge
"""

result = app.invoke({
    "messages": [
        HumanMessage(
            content=query
        )
    ]
})

In [ ]:
for msg in result["messages"]:
    print(msg)